# DuckPD Advanced Features & New Capabilities Walkthrough

Welcome to the **DuckPD Features Walkthrough**! This notebook demonstrates the newest capabilities of DuckPD on **real-world financial market data** using the [AlphaDojo/dojo_stock_news](https://huggingface.co/datasets/AlphaDojo/dojo_stock_news) dataset (~3.9M articles).

### What you will see:
- **Direct Remote Parquet Scanning**: Query millions of rows in cloud parquet without loading full datasets into Python memory.
- **Vectorized String Accessors (`.str`)**: Clean publisher names, extract headlines, and filter topics lazily.
- **Multi-Table Relational Merges (`merge`)**: Join multi-million row news feeds with ticker metadata tables.
- **Multi-Frame Concatenation (`duckpd.concat`)**: Combine filtered partitions with automatic schema union and null-padding.
- **Extended Reductions**: Compute standard deviation (`std`), variance (`var`), median (`median`), and quantiles (`quantile`).
- **Advanced Multi-Column GroupBy**: Named aggregations across publishers and tickers.
- **Window & Positional Transforms**: Cumulative, rank, difference, rolling, expanding, and shifted analytics with guaranteed ordering.
- **Persistence, Query Plans & Direct Parquet Export**: Reuse materialized intermediates, inspect pushdown, and write without pandas fallback.

## 1. Setup Session & Connect to Remote Parquet

Initialize a DuckPD session with custom memory and execution settings, then lazily scan the 3.9M row dataset hosted on Hugging Face.

In [3]:
import pandas as std_pd

import duckpd as pd

print(f"DuckPD Version: {pd.__version__}")
session = pd.connect(memory_limit="1GB", threads=4)

# Remote dataset from AlphaDojo (~3.9M financial news rows)
DATA_URL = "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"

# Lazily scan parquet directly over HTTP with an explicit ordering guarantee
news_df = session.read_parquet(DATA_URL, order_by="publish_date")

print("Lazy DataFrame created:")
print(f"Columns: {news_df.columns}")
print(f"Session executions so far: {session.execution_count}")

DuckPD Version: 0.1.2
Lazy DataFrame created:
Columns: ('title', 'image', 'ago', 'primarysymbol', 'primarytopic', 'publisher', 'url', 'id', 'imagedomain', 'description', 'primarytopic_url', 'publisher_logo', 'publish_date', 'on_symbol_json', 'symbol', 'source')
Session executions so far: 0


## 2. Vectorized String Accessors (`.str`)

Clean publisher names, compute headline lengths, and flag earnings-related announcements lazily using DuckPD's `.str` accessor methods.

In [ ]:
# Perform lazy string feature engineering
enriched_news = news_df.assign(
    publisher_clean=news_df["publisher"].str.strip().str.upper(),
    title_len=news_df["title"].str.len(),
    is_earnings=news_df["title"].str.upper().str.contains("EARNINGS"),
    is_option_activity=news_df["title"].str.contains("Option Activity"),
)

# Inspect a bounded preview pushed down to DuckDB
preview_cols = ["symbol", "publisher_clean", "title_len", "is_earnings", "title"]
enriched_news[preview_cols].head(5)

,symbol,publisher_clean,title_len,is_earnings,title
0,0736.HK,None,20,False,中国置业投资疑遭中车抛弃 单日暴跌82%
1,0137.HK,None,20,False,金辉集团冲刺IPO 拟募60亿开发房地产
2,600069.SS,None,20,False,收购人背景复杂 银鸽投资被上交所二次问询
3,160212.SZ,None,24,False,22家公司修正年报业绩预告 6只机构宠爱股送惊喜
4,160211.SZ,None,24,False,22家公司修正年报业绩预告 6只机构宠爱股送惊喜


## 3. Multi-Table Relational Merging (`merge`)

Join the multi-million row news dataset with a ticker reference metadata table. The join and predicates are compiled into relational SQL execution.

In [5]:
# Reference table for prominent tech & consumer market cap leaders
ticker_meta = session.from_pandas(
    std_pd.DataFrame(
        {
            "symbol": ["AAPL", "NVDA", "MSFT", "AMZN", "TSLA", "GOOGL"],
            "company_name": [
                "Apple Inc.",
                "NVIDIA Corp.",
                "Microsoft Corp.",
                "Amazon.com Inc.",
                "Tesla Inc.",
                "Alphabet Inc.",
            ],
            "sector": [
                "Technology",
                "Semiconductors",
                "Software",
                "E-Commerce",
                "Automotive",
                "Communication",
            ],
            "market_tier": [
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
                "Mega Cap",
            ],
        }
    )
)

# Merge ticker metadata with news stream and sort by publish_date
news_with_sector = ticker_meta.merge(
    enriched_news, on="symbol", how="inner"
).sort_values("publish_date")

news_with_sector[
    ["symbol", "company_name", "sector", "publisher_clean", "title_len", "title"]
].head(5)

,symbol,company_name,sector,publisher_clean,title_len,title
0,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,81,"If You'd Invested $10,000 in Tesla a Decade Ag..."
1,TSLA,Tesla Inc.,Automotive,THE MOTLEY FOOL,128,Cathie Wood's ARK Bought $40 Million of Nvidia...
2,TSLA,Tesla Inc.,Automotive,ZACKS,109,"The Zacks Analyst Blog Highlights SpaceX, Tesl..."
3,TSLA,Tesla Inc.,Automotive,ZACKS,62,Adient Q3 Earnings Miss Estimates on Higher Co...
4,TSLA,Tesla Inc.,Automotive,ZACKS,57,Goodyear Q2 Earnings Miss Expectations on Volu...


## 4. Multi-Frame Concatenation (`duckpd.concat`)

Combine distinct ticker news subsets row-wise with automatic schema union and null-padding.

In [6]:
# Split subsets and enrich one partition with custom category tags
nvda_news = news_with_sector[news_with_sector["symbol"] == "NVDA"].assign(
    focus_area="AI Hardware"
)[["symbol", "company_name", "focus_area", "publisher_clean", "title"]]

tsla_news = news_with_sector[news_with_sector["symbol"] == "TSLA"][
    ["symbol", "company_name", "publisher_clean", "title"]
]

# Concatenate partitions: focus_area will be padded with NULLs for TSLA
combined_stream = pd.concat([nvda_news, tsla_news])
print("Union Columns:", combined_stream.columns)

combined_stream.head(6)

Union Columns: ('symbol', 'company_name', 'focus_area', 'publisher_clean', 'title')


,symbol,company_name,focus_area,publisher_clean,title
0,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,"If I Could Invest in Just 1 ETF in 2026, Here'..."
1,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,"Top Stock Reports for NVIDIA, ExxonMobil & HSBC"
2,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,A Guide to Semiconductor ETFs
3,NVDA,NVIDIA Corp.,AI Hardware,THE MOTLEY FOOL,Palantir Billionaire Peter Thiel Just Put 33% ...
4,NVDA,NVIDIA Corp.,AI Hardware,ZACKS,CBRS' Global Expansion Boosts Growth Prospects...
5,NVDA,NVIDIA Corp.,AI Hardware,NASDAQ.COM,"After Hours Most Active for Aug 28, 2026 : NU..."


## 5. Extended Statistical & Boolean Reductions

Calculate statistical metrics across headline length and content properties (`mean`, `median`, `std`, `var`, `quantile`, `any`, `all`) computed in a single SQL query in DuckDB.

In [7]:
print("--- Headline Length Statistical Metrics ---")
print(f"Mean Length:       {news_with_sector['title_len'].mean():.2f}")
print(f"Median Length:     {news_with_sector['title_len'].median():.2f}")
print(f"Std Deviation:     {news_with_sector['title_len'].std():.2f}")
print(f"Variance:          {news_with_sector['title_len'].var():.2f}")
print(f"25th Percentile:   {news_with_sector['title_len'].quantile(0.25):.2f}")
print(f"75th Percentile:   {news_with_sector['title_len'].quantile(0.75):.2f}")
print(f"95th Percentile:   {news_with_sector['title_len'].quantile(0.95):.2f}")

print("\n--- Boolean Reductions on Filtered Subset ---")
print(f"All headlines mention earnings? {news_with_sector['is_earnings'].all()}")
print(f"Any headline mentions earnings? {news_with_sector['is_earnings'].any()}")

--- Headline Length Statistical Metrics ---
Mean Length:       80.41
Median Length:     72.00
Std Deviation:     30.80
Variance:          948.74
25th Percentile:   60.00
75th Percentile:   99.00
95th Percentile:   142.05

--- Boolean Reductions on Filtered Subset ---
All headlines mention earnings? False
Any headline mentions earnings? True


## 6. Advanced GroupBy & Multi-Metric Aggregations

Perform analytical grouping across publishers and tickers using named aggregations, calculating article volume, average length, dispersion, and extreme values.

In [8]:
# Aggregate news analytics by publisher across top market-cap tickers
publisher_analytics = (
    news_with_sector.groupby(["publisher_clean"], as_index=False)
    .agg(
        article_count=("title", "count"),
        avg_headline_len=("title_len", "mean"),
        std_headline_len=("title_len", "std"),
        max_headline_len=("title_len", "max"),
        min_headline_len=("title_len", "min"),
    )
    .sort_values("article_count", ascending=False)
)

publisher_analytics.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len
0,THE MOTLEY FOOL,762,90.030184,32.936213,204,24
1,ZACKS,260,63.326923,12.015950,109,28
2,MARKETBEAT,65,63.230769,9.380447,81,41
3,RTTNEWS,29,64.655172,12.647066,95,34
4,BARCHART,29,55.448276,10.628755,79,36
5,NASDAQ.COM,28,101.357143,11.823437,111,76
6,BNK INVEST,27,36.703704,10.487409,53,24


## 7. Window & Positional Transforms (`cumsum`, `rank`, `diff`)

Execute analytical window operations over ordered streams. DuckPD ensures window operations execute cleanly in DuckDB without in-memory materialization and validates that explicit `order_by` or `sort_values` guarantees are present.

In [9]:
# Compute cumulative article counts, volume ranks, and incremental step differences
ranked_publishers = publisher_analytics.assign(
    volume_rank=publisher_analytics["article_count"].rank(
        method="dense", ascending=False
    ),
    cumulative_articles=publisher_analytics["article_count"].cumsum(),
    article_step_diff=publisher_analytics["article_count"].diff(-1),
)

ranked_publishers.head(10)

,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff
0,THE MOTLEY FOOL,762,90.030184,32.936213,204,24,1.0,762,502.0
1,ZACKS,260,63.326923,12.015950,109,28,2.0,1022,195.0
2,MARKETBEAT,65,63.230769,9.380447,81,41,3.0,1087,36.0
3,BARCHART,29,55.448276,10.628755,79,36,4.0,1116,0.0
4,RTTNEWS,29,64.655172,12.647066,95,34,4.0,1145,1.0
5,NASDAQ.COM,28,101.357143,11.823437,111,76,5.0,1173,1.0
6,BNK INVEST,27,36.703704,10.487409,53,24,6.0,1200,NaN


## 8. Rolling, Expanding & Shifted Analytics

Build richer ordered analytics with row-based rolling and expanding windows. The transforms stay lazy and compile into DuckDB window expressions; persisting then creates a reusable DuckDB table at an explicit execution boundary.

In [10]:
publisher_windows = ranked_publishers.assign(
    rolling_3_avg_articles=ranked_publishers["article_count"]
    .rolling(3, min_periods=1)
    .mean(),
    expanding_articles=ranked_publishers["article_count"].expanding().sum(),
    previous_publisher_articles=ranked_publishers["article_count"].shift(1),
)

persisted_publishers = publisher_windows.persist("publisher_window_summary")
print(f"Executions after persist: {session.execution_count}")
persisted_publishers.head(10)

Executions after persist: 15


,publisher_clean,article_count,avg_headline_len,std_headline_len,max_headline_len,min_headline_len,volume_rank,cumulative_articles,article_step_diff,rolling_3_avg_articles,expanding_articles,previous_publisher_articles
0,THE MOTLEY FOOL,762,90.030184,32.936213,204,24,1.0,762,502.0,762.000000,762.0,NaN
1,ZACKS,260,63.326923,12.015950,109,28,2.0,1022,195.0,511.000000,1022.0,762.0
2,MARKETBEAT,65,63.230769,9.380447,81,41,3.0,1087,36.0,362.333333,1087.0,260.0
3,BARCHART,29,55.448276,10.628755,79,36,4.0,1116,0.0,118.000000,1116.0,65.0
4,RTTNEWS,29,64.655172,12.647066,95,34,4.0,1145,1.0,41.000000,1145.0,29.0
5,NASDAQ.COM,28,101.357143,11.823437,111,76,5.0,1173,1.0,28.666667,1173.0,29.0
6,BNK INVEST,27,36.703704,10.487409,53,24,6.0,1200,NaN,28.000000,1200.0,28.0


## 9. Plan Inspection (`explain()`) & Direct Export

Inspect the relational query plan generated by DuckPD, including predicate pushdown, joins, and window functions. Then write the window-enriched summary directly to Parquet without routing the full result through pandas.

In [11]:
print("=== Compiled Query Plan with Window Transforms ===")
print(publisher_windows.explain())

# Export the analytical summary directly from DuckDB.
publisher_windows.write_parquet("stock_news_summary.parquet", overwrite=True)
print("\nExported stock_news_summary.parquet directly via DuckDB!")

=== Compiled Query Plan with Window Transforms ===


Fallback boundaries: none (policy=error)
Materialization boundaries: none in the logical plan
Remote source boundaries: none
Source fragments: [{"estimated_transfer_bytes": null, "kind": "parquet", "local_required": ["projection", "aggregation", "join", "window", "sort"], "pushdown_candidates": [], "requested": ["projection", "aggregation", "join", "window", "sort"], "source": "https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"}]
Cross-source movement: [{"estimated_transfer_bytes": null, "kind": "cross_source_join", "left": [{"kind": "pandas", "locations": ["7e759af54ff9442a888cbe38280c9e54"]}], "materializes_in_python": false, "right": [{"kind": "parquet", "locations": ["https://huggingface.co/datasets/AlphaDojo/dojo_stock_news/resolve/main/data.parquet"]}], "strategy": "stream_inputs_to_duckdb"}]
DuckPD logical plan:
{
  "node": "ProjectPlan",
  "input": {
    "node": "ProjectPlan",
    "input": {
      "node": "ProjectPlan",
      "input": {
       